<a href="https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Category: Binary Classification

Rationale: We frame this as a binary decision problem where the objective is to predict whether a specific published article requires an editorial refresh (1) or is performing adequately (0). Because human editing teams operate with fixed weekly bandwidth, this model serves as a prioritization system—sorting high-risk, decaying pages to the top of the queue before traffic loss compounds.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

# Works both in Google Colab and when pulled locally in VS Code / GitHub
data_path = '/content/content_refresh_anonymized.csv' if os.path.exists('/content/content_refresh_anonymized.csv') else '../data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(data_path)
print(f"Total dataset records: {df.shape[0]:,}")
print(f"Total feature columns: {df.shape[1]}")

Total dataset records: 30,000
Total feature columns: 44


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Predicted Target: needs_refresh (Flag: 0 or 1)Label Generation Mechanism: Formulated via a data-driven heuristic proxy. Since manual editorial audits across historical content are incomplete, we define ground truth using short-term performance degradation:$$\text{Proxy Label} = 1 \quad \text{if Organic Sessions decline} \ge 30\% \text{ over a 90-day window}, \quad \text{else } 0$$The model learns early structural and engagement signals that precede this sustained traffic drop, enabling proactive intervention rather than reactive fixes.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect candidate features for target formulation
candidate_cols = [c for c in df.columns if any(term in c.lower() for term in ['traffic', 'views', 'clicks', 'score', 'refresh', 'flag', 'target'])]
print("Potential target/proxy columns detected in dataset:")
print(candidate_cols[:8])

# Check target distribution if target already exists
if 'needs_refresh' in df.columns:
    print("\nTarget Class Breakdown:")
    print(df['needs_refresh'].value_counts(normalize=True).round(4) * 100)
else:
    print("\nNote: 'needs_refresh' proxy target will be generated during preprocessing.")

Potential target/proxy columns detected in dataset:
['clicks_90d', 'pageviews_90d', 'clicks_last_30d', 'clicks_prev_30d', 'ai_traffic_pct']

Note: 'needs_refresh' proxy target will be generated during preprocessing.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric: Recall (Sensitivity) with a minimum operational benchmark of $\ge 0.85$ (85%).Defense: In digital content publishing, the cost of a False Negative (ignoring a page undergoing severe organic decay) is high: permanent loss of search rankings and compounding revenue degradation. Conversely, a False Positive (flagging a stable page) carries a low cost—a quick 60-second review by an editor who clears it. Maximizing Recall ensures we catch almost all fading content while Precision acts as a secondary efficiency constraint.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check dataset volume to evaluate statistical confidence for recall
row_count = df.shape[0]
print(f"Sample size available for model evaluation: {row_count:,} records")
print(f"At 85% recall, estimated false negative tolerance: ~{int(row_count * 0.25 * 0.15):,} missed pages max")

Sample size available for model evaluation: 30,000 records
At 85% recall, estimated false negative tolerance: ~1,125 missed pages max


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: 1 observation = 1 unique Content Page (URL).

Every single record in the tabular dataset captures the historical engagement metrics, age, metadata, and structural features of an individual published page measured across a standardized observation window.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Extract sample slice representing the unit of analysis
sample_view = df.head(5)

# Confirm identity granularity
if 'url' in df.columns:
    print(f"Unique URLs: {df['url'].nunique():,} | Total Rows: {len(df):,}")
elif 'id' in df.columns:
    print(f"Unique Article IDs: {df['id'].nunique():,} | Total Rows: {len(df):,}")

sample_view

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
Limitations of Heuristic / If-Else Rules:

Non-Linear Feature Relationships: Content decay is not purely a function of time. Evergreen articles can maintain traffic for years, whereas trending news items decay in weeks. Hardcoded logic like if age > 180 days triggers massive false alarms.

High-Dimensional Interactions: Traffic shifts depend on the simultaneous interaction of impression decay, click-through rate variations, query volatility, and category seasonality. Handcrafting if/else statements for 10+ interacting signals creates brittle, unmaintainable rules.

Dynamic Decision Thresholds: Machine learning produces probabilistic output scores, letting content managers dynamically adjust risk thresholds based on available weekly editorial staffing.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display distribution variance across numerical features to illustrate non-linear complexity
numeric_features = df.select_dtypes(include=[np.number]).columns[:5]
print("Feature variance illustrating non-linear patterns across content items:")
df[numeric_features].describe().T[['mean', 'std', 'min', '50%', 'max']]

Feature variance illustrating non-linear patterns across content items:


,mean,std,min,50%,max
search_volume,158.882391,1518.270825,0.0,10.0,74000.00
competition,0.146954,0.285241,0.0,0.0,1.00
cpc,0.485342,2.101560,0.0,0.0,100.36
word_count,3107.760325,1452.382598,8.0,2877.0,9546.00
char_count,20665.277835,10115.344042,40.0,19116.0,111158.00


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.